In [8]:
"""
load_ckd_models.py  (v4-2 기준)
═══════════════════════════════════════════════════════════════════════════════
다른 인스턴스에서 저장된 CKD 모델 패키지를 로딩하는 독립 스크립트.

사용법
──────
    python load_ckd_models.py --pkg_dir /home/ubuntu/ckd_output/models
    python load_ckd_models.py --pkg_dir ./models --cpu   # GPU 없는 환경
"""

import argparse, json, sys
from pathlib import Path

import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from tabpfn.model_loading import load_fitted_tabpfn_model


def load_package(pkg_dir: str | Path, map_to_cpu: bool = False) -> dict:
    """
    저장된 모델 패키지 전체를 로딩합니다.

    Returns
    -------
    {
      "models_A"  : dict[model_name → Pipeline],
      "models_B"  : dict[model_name → Pipeline],
      "imputer_A" : TwoTierImputer,
      "imputer_B" : TwoTierImputer,
      "metadata"  : dict,
    }
    """
    pkg = Path(pkg_dir)

    # ── Step 1. TwoTierImputer 클래스 등록 (joblib.load 전 필수) ─────────────
    src_dir = str(pkg / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    try:
        import two_tier_imputer as _tti
        import __main__
        __main__.TwoTierImputer = _tti.TwoTierImputer
        print(f"[LOAD] TwoTierImputer: {_tti.__file__} ✅")
    except ImportError as e:
        raise ImportError(
            f"two_tier_imputer.py를 찾을 수 없습니다.\n"
            f"  탐색 경로: {src_dir}\n"
            f"  해결: {src_dir} 에 two_tier_imputer.py가 있는지 확인하세요.\n"
            f"  오류: {e}"
        ) from e

    # ── Step 2. metadata ──────────────────────────────────────────────────────
    with open(pkg / "metadata.json", encoding="utf-8") as f:
        metadata = json.load(f)
    model_order = metadata["model_order"]
    model_specs = metadata["model_specs"]
    print(f"[LOAD] metadata: {len(model_order)} models | "
          f"horizon={metadata.get('horizon_days')} days")
    if "key_package_versions" in metadata:
        print(f"[LOAD] 저장 당시 패키지: "
              f"{list(metadata['key_package_versions'].values())}")

    # ── Step 3. 모델 로딩 ─────────────────────────────────────────────────────
    result = {"models_A": {}, "models_B": {}, "metadata": metadata}

    for scen_tag, key in [("scenario_A", "models_A"), ("scenario_B", "models_B")]:
        scen_dir = pkg / scen_tag
        if not scen_dir.exists():
            print(f"[LOAD] ⚠️  {scen_tag} 폴더 없음 — 건너뜁니다")
            continue

        for name in model_order:
            pre_path = scen_dir / "preprocess" / f"{name}_preprocess.joblib"
            tpf_path = scen_dir / "tabpfn_fit"  / f"{name}.tabpfn_fit"
            if not pre_path.exists() or not tpf_path.exists():
                print(f"[LOAD] ⚠️  {scen_tag}/{name} 파일 없음 — 건너뜁니다")
                continue

            pre = joblib.load(pre_path)
            clf = load_fitted_tabpfn_model(tpf_path)
            if map_to_cpu and hasattr(clf, "to"):
                clf = clf.to("cpu")

            result[key][name] = Pipeline(steps=[("preprocess", pre), ("clf", clf)])
            print(f"[LOAD] ✅ {scen_tag}/{name}")

        imp_path = scen_dir / "imputer.joblib"
        if imp_path.exists():
            imp     = joblib.load(imp_path)
            imp_key = "imputer_A" if scen_tag == "scenario_A" else "imputer_B"
            result[imp_key] = imp
            print(f"[LOAD] ✅ {scen_tag}/imputer  (include_cap={imp.include_cap})")

    return result


def predict(
    pkg: dict,
    df_new: pd.DataFrame,
    scenario: str = "B",
    model_name: str = "base_lsm_cap",
) -> np.ndarray:
    """
    로딩된 패키지로 새 데이터 예측 확률을 반환합니다.

    전처리 순서 (자동 처리):
        1. add_fibroscan_date_numeric()  ← Scenario B only
        2. imputer.transform()
        3. pipeline.predict_proba()

    Returns
    -------
    np.ndarray, shape (n_samples,)  — CKD 발생 예측 확률
    """
    from two_tier_imputer import add_fibroscan_date_numeric

    imp      = pkg[f"imputer_{scenario}"]
    pipe     = pkg[f"models_{scenario}"][model_name]
    features = pkg["metadata"]["model_specs"][model_name]

    df_prep = df_new.copy()
    if scenario == "B" and "Fibroscan_Date_numeric" not in df_prep.columns:
        df_prep = add_fibroscan_date_numeric(df_prep)

    df_imp = imp.transform(df_prep)
    return pipe.predict_proba(df_imp[features])[:, 1]


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="CKD 모델 패키지 로더")
    parser.add_argument("--pkg_dir", required=True, help="models/ 디렉터리 경로")
    parser.add_argument("--cpu", action="store_true", help="GPU→CPU 리매핑")
    sys.argv = ['', '--pkg_dir', '/home/ubuntu/ckd_output/models']
    args = parser.parse_args()

    pkg = load_package(args.pkg_dir, map_to_cpu=args.cpu)

    print("\n[SUMMARY] 로딩 완료")
    for scen in ["A", "B"]:
        key = f"models_{scen}"
        if key in pkg and pkg[key]:
            print(f"  Scenario {scen}: {list(pkg[key].keys())}")

[LOAD] TwoTierImputer: /home/ubuntu/20260406_TabPFN_v2/CKD/two_tier_imputer.py ✅
[LOAD] metadata: 6 models | horizon=1826 days
[LOAD] 저장 당시 패키지: ['numpy==2.3.5', 'pandas==2.3.3', 'scikit-learn==1.8.0', 'tabpfn==6.4.1', 'tabpfn-common-utils==0.2.16', 'torch==2.7.1+cu118', 'torchaudio==2.7.1+cu118', 'torchvision==0.22.1+cu118']
[LOAD] ✅ scenario_A/base
[LOAD] ✅ scenario_A/base_lsm
[LOAD] ✅ scenario_A/base_lsm_cap
[LOAD] ✅ scenario_A/base_inflam
[LOAD] ✅ scenario_A/base_inflam_lsm
[LOAD] ✅ scenario_A/base_inflam_lsm_cap
[LOAD] ✅ scenario_A/imputer  (include_cap=False)
[LOAD] ✅ scenario_B/base
[LOAD] ✅ scenario_B/base_lsm
[LOAD] ✅ scenario_B/base_lsm_cap
[LOAD] ✅ scenario_B/base_inflam
[LOAD] ✅ scenario_B/base_inflam_lsm
[LOAD] ✅ scenario_B/base_inflam_lsm_cap
[LOAD] ✅ scenario_B/imputer  (include_cap=True)

[SUMMARY] 로딩 완료
  Scenario A: ['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']
  Scenario B: ['base', 'base_lsm', 'base_lsm_cap', 'base_inf

In [9]:
import joblib
joblib.dump({
    'pred_test_A': pred_test_A, 'pred_test_B': pred_test_B,
    'pred_ext_A':  pred_ext_A,  'pred_ext_B':  pred_ext_B,
    'y_test_A':    y_test_A,    'y_test':      y_test,
    'y_ext_A':     y_ext_A,     'y_ext':       y_ext,
}, '/home/ubuntu/ckd_output/predictions_cache.joblib')

NameError: name 'pred_test_A' is not defined

In [10]:
%whos

Variable                   Type              Data/Info
------------------------------------------------------
Path                       type              <class 'pathlib.Path'>
Pipeline                   ABCMeta           <class 'sklearn.pipeline.Pipeline'>
TwoTierImputer             type              <class 'two_tier_imputer.TwoTierImputer'>
argparse                   module            <module 'argparse' from '<...>/python3.12/argparse.py'>
args                       Namespace         Namespace(pkg_dir='/home/<...>utput/models', cpu=False)
joblib                     module            <module 'joblib' from '/h<...>ages/joblib/__init__.py'>
json                       module            <module 'json' from '/usr<...>on3.12/json/__init__.py'>
key                        str               models_B
load_fitted_tabpfn_model   function          <function load_fitted_tab<...>_model at 0x71e9cdf509a0>
load_package               function          <function load_package at 0x71e9ccd88040>
np      

In [11]:
print(pkg.keys())

dict_keys(['models_A', 'models_B', 'metadata', 'imputer_A', 'imputer_B'])


In [12]:
import inspect
print(inspect.getsource(predict))

def predict(
    pkg: dict,
    df_new: pd.DataFrame,
    scenario: str = "B",
    model_name: str = "base_lsm_cap",
) -> np.ndarray:
    """
    로딩된 패키지로 새 데이터 예측 확률을 반환합니다.

    전처리 순서 (자동 처리):
        1. add_fibroscan_date_numeric()  ← Scenario B only
        2. imputer.transform()
        3. pipeline.predict_proba()

    Returns
    -------
    np.ndarray, shape (n_samples,)  — CKD 발생 예측 확률
    """
    from two_tier_imputer import add_fibroscan_date_numeric

    imp      = pkg[f"imputer_{scenario}"]
    pipe     = pkg[f"models_{scenario}"][model_name]
    features = pkg["metadata"]["model_specs"][model_name]

    df_prep = df_new.copy()
    if scenario == "B" and "Fibroscan_Date_numeric" not in df_prep.columns:
        df_prep = add_fibroscan_date_numeric(df_prep)

    df_imp = imp.transform(df_prep)
    return pipe.predict_proba(df_imp[features])[:, 1]



In [ ]:
import sys
from sklearn.model_selection import train_test_split
sys.path.insert(0, '/home/ubuntu/ckd_output/models/src')
from two_tier_imputer import is_cap_era, add_fibroscan_date_numeric, coerce_numeric

# ── 상수 ──────────────────────────────────────────────────────────────────
HOSP_INTERNAL = 10
HOSP_EXTERNAL = 20
RANDOM_STATE  = 42
MAX_TABPFN_N  = 10000

# ── 1. 데이터 로드 ─────────────────────────────────────────────────────────
df_raw = pd.read_excel('/home/ubuntu/Fibroscan_CKD_2025.xlsx')

# ── 2. notebook 1과 동일한 전처리 ─────────────────────────────────────────
# df_final 생성 과정: Follow-up 있는 환자만, y 라벨 생성
df_final = df_raw[df_raw['Follow-up_duration'].notna()].copy()
df_final['y'] = ((df_final['CKD'] == 1) & (df_final['TTO_CKD'] <= 1825)).astype(int)

# ── 3. 분리 ───────────────────────────────────────────────────────────────
internal = df_final[df_final['Hospital'] == HOSP_INTERNAL].copy()
external = df_final[df_final['Hospital'] == HOSP_EXTERNAL].copy()

idx_train, idx_test = train_test_split(
    internal.index.values, test_size=0.2,
    random_state=RANDOM_STATE, stratify=internal['y'].values
)
df_train = internal.loc[idx_train].copy()
df_test  = internal.loc[idx_test].copy()
df_ext   = external.copy()

y_test = df_test['y'].astype(int).values
y_ext  = df_ext['y'].astype(int).values

# ── 4. Scenario A: CAP-era만 ──────────────────────────────────────────────
df_test_A = df_test[is_cap_era(df_test)].copy()
df_ext_A  = df_ext[is_cap_era(df_ext)].copy()
y_test_A  = df_test_A['y'].astype(int).values
y_ext_A   = df_ext_A['y'].astype(int).values

# ── 5. 예측 생성 ──────────────────────────────────────────────────────────
MODEL_ORDER = ['base', 'base_lsm', 'base_lsm_cap',
               'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']

pred_test_A = {m: predict(pkg, df_test_A, scenario='A', model_name=m) for m in MODEL_ORDER}
pred_ext_A  = {m: predict(pkg, df_ext_A,  scenario='A', model_name=m) for m in MODEL_ORDER}
pred_test_B = {m: predict(pkg, df_test,   scenario='B', model_name=m) for m in MODEL_ORDER}
pred_ext_B  = {m: predict(pkg, df_ext,    scenario='B', model_name=m) for m in MODEL_ORDER}

print("[OK] 예측 완료")
print(f"  test_A n={len(y_test_A)}, ext_A n={len(y_ext_A)}")
print(f"  test_B n={len(y_test)},   ext_B n={len(y_ext)}")